# ASSTF Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SafewareTaiwan/leanai-asstf/blob/main/notebooks/ASSTF_Quickstart.ipynb)

Run ASSTF in under 5 minutes. This notebook shows how to:
1. Install ASSTF
2. Replace `nn.Linear` with `ASSTFLinear`
3. Train with bilevel optimization
4. Adapt at inference time with `SurpriseMinimizer`

In [ ]:
# Install ASSTF
!pip install leanai-asstf

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from asstf import ASSTFLinear, BilevelTrainer, SurpriseMinimizer, count_parameters

torch.manual_seed(42)

## 1. Build a tiny ASSTF classifier

In [ ]:
class TinyASSTFClassifier(nn.Module):
    def __init__(self, in_dim=64, hidden=32, num_classes=4):
        super().__init__()
        self.fc1 = ASSTFLinear(in_dim, hidden, structural_rank=4)
        self.fc2 = ASSTFLinear(hidden, num_classes, structural_rank=2)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = TinyASSTFClassifier()
print(f"Parameters: {count_parameters(model)[0]:,}")

x = torch.randn(8, 64)
logits = model(x)
print(logits.shape)

## 2. Train with Bilevel Optimization

In [ ]:
# Synthetic dataset
X_train = torch.randn(256, 64)
y_train = torch.randint(0, 4, (256,))
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train, y_train), batch_size=32, shuffle=True
)

trainer = BilevelTrainer(model, lr_core=1e-3, lr_struct=5e-4, alternate=True)

for epoch in range(10):
    loss = trainer.train_epoch(model, train_loader, F.cross_entropy)
    if epoch % 2 == 0:
        print(f"Epoch {epoch}: loss={loss:.4f}")

## 3. Adapt at Inference Time

In [ ]:
# New domain data (simulated shift)
X_new = torch.randn(16, 64) * 0.5 + 0.3
y_new = torch.randint(0, 4, (16,))

adapter = SurpriseMinimizer(model, lr=1e-4, max_steps=5)

# Before adaptation
model.eval()
with torch.no_grad():
    before = model(X_new).argmax(dim=1)
    acc_before = (before == y_new).float().mean().item()
    print(f"Accuracy before adaptation: {acc_before:.2%}")

# Adapt structural parameters on new domain
for x_batch, y_batch in torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_new, y_new), batch_size=8
):
    adapter.adapt(model, x_batch, target=y_batch)

# After adaptation
model.eval()
with torch.no_grad():
    after = model(X_new).argmax(dim=1)
    acc_after = (after == y_new).float().mean().item()
    print(f"Accuracy after adaptation: {acc_after:.2%}")

## Next Steps

- Explore the [full documentation](https://safewaretaiwan.github.io/leanai-asstf/)
- Run the [6 application demos](https://github.com/SafewareTaiwan/leanai-asstf/tree/main/app_01_gesture)
- Read the [architecture deep-dive](https://github.com/SafewareTaiwan/leanai-asstf/blob/main/docs/ARCHITECTURE_EN.md)